# UIT DSC 2026 — LegalQA v9.1 trên Kaggle T4
Luồng VRAM tuần tự: Embedding/global dense + official train-memory → unload → Reranker → unload → Qwen base hoặc QLoRA adapter → unload → ViLegalBERT → submission.zip. Độ dài được học từ official train; BARTpho không thuộc runtime.

In [ ]:
DEVICE = 'cuda:0'
EMBEDDING_BATCH_SIZE = 2
RERANKER_BATCH_SIZE = 1
VERIFIER_BATCH_SIZE = 4
RERANKER_MAX_LENGTH = 512
MAX_CONTEXT_CHARS = 18000
MAX_NEW_TOKENS = 900
BM25_TOP_K = 140
DENSE_TOP_K = 70
FUSED_TOP_K = 48
RERANK_TOP_K = 24
EVIDENCE_TOP_K = 5
FORCE_INDEX = False
REUSE_RETRIEVAL = True
MODEL_SMOKE = True     # Test thật 4 model runtime; BARTpho đã bị loại
FORCE_DENSE_INDEX = False
DENSE_INDEX_ONLY = False  # True cho session 1; session 2 attach index rồi đổi False
DENSE_INDEX_DIR = '/kaggle/working/legalqa_dense_index'
TRAIN_MEMORY = True       # kNN/few-shot/candidate chỉ từ official train
CHECKPOINT_EVERY = 50  # Lưu recovery ZIP sau mỗi 50 câu mới
AUTO_RESUME = True     # Tự tìm legalqa_recovery_latest.zip trong Kaggle Input
RECOVERY_OUTPUT = '/kaggle/working/legalqa_recovery_latest.zip'

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys
input_root = Path('/kaggle/input')
public_files = sorted(input_root.rglob('public-official.json'))
if len(public_files) != 1:
    raise FileNotFoundError(f'Cần duy nhất một public-official.json: {public_files}')
data_root = public_files[0].parent
train_file = data_root / 'train.json'
if not train_file.is_file():
    raise FileNotFoundError(f'Thiếu official train.json cạnh public: {train_file}')
corpus_dirs = [data_root / 'selected-contexts'] if (data_root / 'selected-contexts').is_dir() else []
if not corpus_dirs:
    context_files = list(data_root.rglob('context_*.json'))
    parent_counts = {}
    for path in context_files:
        parent_counts[path.parent] = parent_counts.get(path.parent, 0) + 1
    corpus_dirs = [max(parent_counts, key=parent_counts.get)] if parent_counts else []
if len(corpus_dirs) != 1:
    raise FileNotFoundError(f'Không xác định được corpus: {corpus_dirs}')
def is_inference_source(path):
    return (path / 'src' / 'legalqa').is_dir() and (path / 'kaggle' / 'run_inference_only.py').is_file() and (path / 'configs' / 'kaggle_inference_t4.yaml').is_file()
source_projects = sorted({p.parent for p in input_root.rglob('pyproject.toml') if is_inference_source(p.parent)})
if len(source_projects) != 1:
    raise RuntimeError(f'Cần đúng một source inference-only mới: {source_projects}')
selected_source = source_projects[0]
source_dir = Path('/kaggle/working/legalqa_source')
if source_dir.exists():
    shutil.rmtree(source_dir)
shutil.copytree(selected_source, source_dir)
project = source_dir
wheels = sorted(str(path) for path in (project / 'wheelhouse').glob('*.whl'))
required = ('transformers-5.14.1-', 'huggingface_hub-1.26.1-', 'hf_xet-1.6.0-', 'click-8.4.2-', 'regex-2026.7.19-', 'tokenizers-0.22.2-', 'accelerate-1.14.0-', 'safetensors-0.8.0-', 'sentence_transformers-5.7.0-', 'peft-0.18.1-', 'bitsandbytes-0.49.2-', 'sentencepiece-0.2.1-')
missing = [prefix for prefix in required if not any(Path(wheel).name.startswith(prefix) for wheel in wheels)]
if missing:
    raise FileNotFoundError(f'Wheelhouse offline chưa đầy đủ: {missing}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--upgrade', *wheels], check=True)
src_path = str(project / 'src')
sys.path.insert(0, src_path)
os.environ['PYTHONPATH'] = src_path + os.pathsep + os.environ.get('PYTHONPATH', '')
os.environ['PYTHONUTF8'] = '1'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv'], check=True)
print('Public:', public_files[0])
print('Train:', train_file)
print('Corpus:', corpus_dirs[0])
print('Model configs:', len(list(input_root.rglob('config.json'))))
print('Source:', selected_source)

In [ ]:
dense_index_candidates = []
for manifest_path in input_root.rglob('manifest.json'):
    try:
        payload = json.loads(manifest_path.read_text(encoding='utf-8'))
    except Exception:
        continue
    if payload.get('schema') == 'global-dense-v1' and payload.get('status') == 'complete':
        dense_index_candidates.append(manifest_path.parent)
if len(dense_index_candidates) > 1:
    raise RuntimeError(f'Chỉ attach tối đa một global dense index: {dense_index_candidates}')
dense_index_input = dense_index_candidates[0] if dense_index_candidates else None
adapter_candidates = sorted({p.parent for p in input_root.rglob('training_complete.json') if (p.parent / 'adapter_config.json').is_file()})
if len(adapter_candidates) > 1:
    raise RuntimeError(f'Chỉ attach một completed Qwen adapter: {adapter_candidates}')
generator_adapter = adapter_candidates[0] if adapter_candidates else None
weight_candidates = sorted(input_root.rglob('tuned_selection.yaml'))
if len(weight_candidates) > 1:
    raise RuntimeError(f'Chỉ attach một tuned_selection.yaml: {weight_candidates}')
selection_weights = weight_candidates[0] if weight_candidates else None
recovery_bundles = sorted(input_root.rglob('legalqa_recovery_latest.zip')) if AUTO_RESUME else []
if len(recovery_bundles) > 1:
    raise RuntimeError(f'Chỉ attach đúng một recovery ZIP: {recovery_bundles}')
resume_bundle = recovery_bundles[0] if recovery_bundles else None
print('Resume bundle:', resume_bundle or 'Không có — chạy mới')
print('Global dense index:', dense_index_input or 'Chưa có — sẽ dựng checkpointable trong working')
print('Qwen adapter:', generator_adapter or 'Không có — dùng base')
print('Selector weights:', selection_weights or 'Không có — dùng config mặc định')
command = [
    sys.executable, str(project / 'kaggle' / 'run_inference_only.py'),
    '--config', str(project / 'configs' / 'kaggle_inference_t4.yaml'),
    '--input-root', '/kaggle/input',
    '--working', '/kaggle/working/legalqa_inference',
    '--submission-output', '/kaggle/working/submission.zip',
    '--recovery-output', RECOVERY_OUTPUT,
    '--checkpoint-every', str(CHECKPOINT_EVERY),
    '--device', DEVICE,
    '--embedding-batch-size', str(EMBEDDING_BATCH_SIZE),
    '--reranker-batch-size', str(RERANKER_BATCH_SIZE),
    '--verifier-batch-size', str(VERIFIER_BATCH_SIZE),
    '--reranker-max-length', str(RERANKER_MAX_LENGTH),
    '--max-context-chars', str(MAX_CONTEXT_CHARS),
    '--max-new-tokens', str(MAX_NEW_TOKENS),
    '--bm25-top-k', str(BM25_TOP_K),
    '--dense-top-k', str(DENSE_TOP_K),
    '--fused-top-k', str(FUSED_TOP_K),
    '--rerank-top-k', str(RERANK_TOP_K),
    '--evidence-top-k', str(EVIDENCE_TOP_K),
    '--dense-index-dir', DENSE_INDEX_DIR,
    '--model-smoke' if MODEL_SMOKE else '--no-model-smoke',
    '--reuse-retrieval' if REUSE_RETRIEVAL else '--no-reuse-retrieval',
    '--train-memory' if TRAIN_MEMORY else '--no-train-memory',
]
if dense_index_input is not None:
    command.extend(['--dense-index-input', str(dense_index_input)])
if resume_bundle is not None:
    command.extend(['--resume-bundle', str(resume_bundle)])
if generator_adapter is not None:
    command.extend(['--generator-adapter', str(generator_adapter)])
if selection_weights is not None:
    command.extend(['--selection-weights', str(selection_weights)])
if DENSE_INDEX_ONLY:
    command.append('--dense-index-only')
if FORCE_DENSE_INDEX:
    command.append('--force-dense-index')
if FORCE_INDEX:
    command.append('--force-index')
print(' '.join(command))
subprocess.run(command, check=True)

In [ ]:
import json, zipfile
submission = Path('/kaggle/working/submission.zip')
recovery = Path(RECOVERY_OUTPUT)
if DENSE_INDEX_ONLY:
    dense_manifest = Path(DENSE_INDEX_DIR) / 'manifest.json'
    assert dense_manifest.exists(), 'Dense index chưa hoàn tất; state.json/vectors.f16 là checkpoint nội bộ.'
    dense_state = json.loads(dense_manifest.read_text(encoding='utf-8'))
    assert dense_state.get('status') == 'complete'
    if TRAIN_MEMORY:
        memory_manifest = Path(DENSE_INDEX_DIR) / 'train_memory' / 'manifest.json'
        assert memory_manifest.exists(), 'Thiếu completed official train-memory index.'
    print('DENSE INDEX READY:', DENSE_INDEX_DIR, dense_state)
    print('Hãy Save Version, tạo Kaggle Dataset từ thư mục này, attach vào session 2 và đặt DENSE_INDEX_ONLY=False.')
else:
    assert recovery.exists(), 'Thiếu recovery ZIP'
    with zipfile.ZipFile(recovery) as archive:
        state = json.loads(archive.read('recovery_state.json').decode('utf-8'))
    print('RECOVERY:', recovery, state)
    assert submission.exists(), 'Inference chưa hoàn tất; hãy lưu recovery ZIP thành Kaggle Dataset rồi chạy session kế tiếp.'
    with zipfile.ZipFile(submission) as archive:
        assert archive.namelist() == ['submission.json']
        payload = json.loads(archive.read('submission.json').decode('utf-8'))
    assert len(payload) == len(json.loads(public_files[0].read_text(encoding='utf-8')))
    assert all(isinstance(value, dict) and isinstance(value.get('answer'), str) for value in payload.values())
    print(f'READY: {submission} — {submission.stat().st_size:,} bytes — {len(payload)} answers')